In [1]:
# Get daily constraint ranked by the abs RT - DA 
from functions import (
    get_recent_constraint_mvalue,
    get_node_dfax_from_constraint_num,
    get_hourly_mvalue_for_constraint_num,
    get_constraint_node_prices_from_constraint_num,
    get_constraints_node_price,
    get_metrics_for_nodes,
    get_recent_direction_accuracy,
    get_weather_date,
    predict_tomorrow_percentile,
    analyze_constraint_by_zone,
    plot_constraint_seasonality,
    plot_recent_hourly_distribution,
)
import pandas as pd

In [2]:
now = pd.Timestamp.now(tz='US/Central')
days_ahead = 2 if now.hour >= 10 else 1
bid_dt = (now + pd.Timedelta(days=days_ahead)).strftime('%Y-%m-%d')
print(bid_dt)

2026-06-27


In [3]:
dt = (pd.Timestamp(bid_dt) - pd.Timedelta(days=1)).strftime('%Y-%m-%d')
mvalue= get_recent_constraint_mvalue(dt, dt, threshold=500)
dfax= get_node_dfax_from_constraint_num(mvalue)
hourly_mvalue = get_hourly_mvalue_for_constraint_num(mvalue)
fundamentals = get_weather_date(hourly_mvalue, dt)

start_dt: 2026-06-26, end_dt: 2026-06-26, Market: SPP
Fetching RT mvalues...
Fetching DA mvalues...
RT constraints: 6, DA constraints: 34
Fetching constraint details...


,oops_constraint_num,rt_total,da_total,rt_da,abs_rt_da_diff,monitored,contingency
0,797451,0.0000,-3726.0122,3726.0122,3726.0122,lnpenn_tap-devilsl,gre:ramseybaltagre:230:1:8
1,775436,0.0000,-1749.5039,1749.5039,1749.5039,xfmrsiouxcy-siouxcy,waue:siouxcyxfkv5a3:23016113.8
2,649753,0.0000,-1323.7881,1323.7881,1323.7881,lnrussett-sbrown,okge:brown2bodlecaney1:138:7:34
3,885392,0.0000,-784.4919,784.4919,784.4919,xfmrwashit1-washit1,wfeccsws:washit1sw_sta:138:1:8
4,776592,0.0000,-717.7345,717.7345,717.7345,xfmrduncan-duncan,wfec:anadarkofletch3comanch2:138
5,567286,0.0000,-692.1220,692.1220,692.1220,lnsugr_crk-sub_f_rev23,indn:sub_csub_n:69:3:12
7,653411,-195.0713,0.0000,-195.0713,195.0713,lnrussett-sbrown,okge:brown2bodlecaney1:138:7:34


RT active constraint-days: 363
DA active constraint-days: 779
   oops_constraint_num          dt  hr  rt_mvalue  da_mvalue    rt_da
0               567286  2025-06-01  17        0.0   -53.6100  53.6100
1               567286  2025-06-01  18        0.0   -60.6793  60.6793
2               567286  2025-06-01  19        0.0   -84.2669  84.2669
3               567286  2025-06-11  13        0.0    -3.6360   3.6360
4               567286  2025-06-11  14        0.0   -49.6189  49.6189

Merged hourly rows: 10282


,oops_constraint_num,dt,hr,rt_mvalue,da_mvalue,rt_da,monitored,contingency
0,567286,2025-06-01,17,0.0,-53.6100,53.6100,lnsugr_crk-sub_f_rev23,indn:sub_csub_n:69:3:12
1,567286,2025-06-01,18,0.0,-60.6793,60.6793,lnsugr_crk-sub_f_rev23,indn:sub_csub_n:69:3:12
2,567286,2025-06-01,19,0.0,-84.2669,84.2669,lnsugr_crk-sub_f_rev23,indn:sub_csub_n:69:3:12
3,567286,2025-06-11,13,0.0,-3.6360,3.6360,lnsugr_crk-sub_f_rev23,indn:sub_csub_n:69:3:12
4,567286,2025-06-11,14,0.0,-49.6189,49.6189,lnsugr_crk-sub_f_rev23,indn:sub_csub_n:69:3:12
...,...,...,...,...,...,...,...,...
10277,885392,2026-06-26,10,0.0,-25.2692,25.2692,xfmrwashit1-washit1,wfeccsws:washit1sw_sta:138:1:8
10278,885392,2026-06-26,21,0.0,-23.2406,23.2406,xfmrwashit1-washit1,wfeccsws:washit1sw_sta:138:1:8
10279,885392,2026-06-26,22,0.0,-119.5320,119.5320,xfmrwashit1-washit1,wfeccsws:washit1sw_sta:138:1:8
10280,885392,2026-06-26,23,0.0,-55.5887,55.5887,xfmrwashit1-washit1,wfeccsws:washit1sw_sta:138:1:8


Reserve zones available: [1, 2, 3, 4, 5, 21]
Wind cols: ['rz1_spp_res_zonal_wind_forecast_f', 'rz2_spp_res_zonal_wind_forecast_f', 'rz3_spp_res_zonal_wind_forecast_f', 'rz4_spp_res_zonal_wind_forecast_f', 'rz5_spp_res_zonal_wind_forecast_f', 'rz21_spp_res_zonal_wind_forecast_f']
Load cols: ['rz1_spp_res_zonal_load_forecast_f', 'rz2_spp_res_zonal_load_forecast_f', 'rz3_spp_res_zonal_load_forecast_f', 'rz4_spp_res_zonal_load_forecast_f', 'rz5_spp_res_zonal_load_forecast_f', 'rz21_spp_res_zonal_load_forecast_f']


In [4]:
# from IPython.display import display, HTML
# row_counts = hourly_mvalue.groupby('monitored').size().sort_values()
# print(row_counts.to_string())

# for name in row_counts.index:
#     display(HTML(f'''
#     <hr style="border: 3px solid black; margin: 30px 0;">
#     <h1 style="background:#2c3e50; color:white; padding:12px 20px; border-radius:6px; font-family:monospace;">
#         {name} &nbsp;<span style="font-size:0.6em; color:#aaa;">({row_counts[name]:,} rows)</span>
#     </h1>
#     <hr style="border: 1px solid #aaa; margin-bottom: 20px;">
#     '''))

#     analyze_constraint_by_zone(fundamentals, hourly_mvalue, name, 4, 95)
#     plot_constraint_seasonality(hourly_mvalue, name)
#     plot_recent_hourly_distribution(hourly_mvalue, name, bid_dt, days=10)
#     table = get_constraints_node_price(hourly_mvalue, dfax, [name])
#     get_metrics_for_nodes(table)

In [5]:
# name = 'lnraun-tekamho'
# zone= 1
# analyze_constraint_by_zone(fundamentals, hourly_mvalue, name, zone, 90)
# plot_constraint_seasonality(hourly_mvalue, name)
# table = get_constraints_node_price(hourly_mvalue, dfax, [name])
# get_metrics_for_nodes(table)

In [6]:
fundamentals = get_weather_date(hourly_mvalue, bid_dt)

Reserve zones available: [1, 2, 3, 4, 5, 21]
Wind cols: ['rz1_spp_res_zonal_wind_forecast_f', 'rz2_spp_res_zonal_wind_forecast_f', 'rz3_spp_res_zonal_wind_forecast_f', 'rz4_spp_res_zonal_wind_forecast_f', 'rz5_spp_res_zonal_wind_forecast_f', 'rz21_spp_res_zonal_wind_forecast_f']
Load cols: ['rz1_spp_res_zonal_load_forecast_f', 'rz2_spp_res_zonal_load_forecast_f', 'rz3_spp_res_zonal_load_forecast_f', 'rz4_spp_res_zonal_load_forecast_f', 'rz5_spp_res_zonal_load_forecast_f', 'rz21_spp_res_zonal_load_forecast_f']


In [7]:
tomorrow_pct = predict_tomorrow_percentile(fundamentals, bid_dt)


--- Zone 1 ---


,hr,wind_value,wind_pct,load_value,load_pct
0,1,1447.5,61.3,4115.9,41.9
1,2,1510.1,64.5,4041.6,58.1
2,3,1479.3,67.7,3955.5,48.4
3,4,1522.8,71.0,3950.5,58.1
4,5,1610.4,87.1,3909.0,45.2
5,6,1661.8,93.5,3975.9,41.9
6,7,1698.1,96.8,4004.5,25.8
7,8,1726.2,96.8,4216.5,32.3
8,9,1792.9,96.8,4325.1,19.4
9,10,1874.5,93.5,4512.8,22.6



--- Zone 2 ---


,hr,wind_value,wind_pct,load_value,load_pct
0,1,3475.2,35.5,744.5,80.6
1,2,3509.7,41.9,731.9,93.5
2,3,3387.8,45.2,716.7,96.8
3,4,3285.8,45.2,709.2,96.8
4,5,3285.1,48.4,700.6,90.3
5,6,3303.6,48.4,707.4,71.0
6,7,3329.6,54.8,717.2,58.1
7,8,3351.6,64.5,729.0,64.5
8,9,3193.1,64.5,757.7,64.5
9,10,3171.5,71.0,809.9,83.9



--- Zone 3 ---


...


--- Zone 5 ---


,hr,wind_value,wind_pct,load_value,load_pct
0,1,3327.0,96.8,3676.4,77.4
1,2,3352.3,96.8,3639.3,90.3
2,3,3377.8,96.8,3630.3,93.5
3,4,3409.3,100.0,3630.6,96.8
4,5,3411.2,100.0,3614.4,87.1
5,6,3435.1,100.0,3606.0,58.1
6,7,3454.1,100.0,3616.1,25.8
7,8,3456.0,100.0,3650.3,16.1
8,9,3472.9,100.0,3756.3,19.4
9,10,3469.2,100.0,3896.9,22.6



--- Zone 21 ---


,hr,wind_value,wind_pct,load_value,load_pct
0,1,892.7,96.8,3139.7,71.0
1,2,901.4,93.5,2939.6,67.7
2,3,882.3,96.8,2800.4,67.7
3,4,845.9,96.8,2730.3,67.7
4,5,822.0,93.5,2668.8,67.7
5,6,811.6,93.5,2654.3,71.0
6,7,788.6,93.5,2706.9,74.2
7,8,725.1,83.9,2800.7,77.4
8,9,628.2,77.4,2917.0,77.4
9,10,560.9,67.7,2997.0,77.4



--- Summary (median pct by zone / peak) ---
1: ow: 93, fw: 79, ol: 74, fl: 46
2: ow: 82, fw: 46, ol: 87, fl: 91
3: ow: 88, fw: 56, ol: 96, fl: 98
4: ow: 93, fw: 67, ol: 83, fl: 88
5: ow: 100, fw: 98, ol: 67, fl: 87
21: ow: 80, fw: 93, ol: 50, fl: 67
